# Step 06 — Chain Prompting

🇬🇧 **English** (this notebook) · 🇩🇪 Deutsch — not yet available.

Break the task into two sequential API calls. The output of the first call becomes the input for the second — a deliberate split into two narrower sub-tasks, instead of one bigger prompt.

## Learning objective

By the end of this notebook, you will:

- Understand what "chain prompting" means: passing one call's output into the next call's input
- Have wired two sequential `llm.call(...)` invocations together by hand
- Be able to judge whether splitting a task into stages actually improves the final answer, or just adds latency

## Prerequisites

- [Step 03 — Zero-Shot Prompting](step_03_zero_shot_prompting.ipynb) and [Step 05 — Prompt Template](step_05_prompt_template.ipynb) completed
- The same `.env` setup as the previous steps

## Background

Steps 04 and 05 both used **one** API call — richer prompts, but still one round trip. Chain prompting instead uses **two or more** calls, where one call's output becomes the next call's input, plain text pasted into a new `user` message. Each call is independent: no conversation history carries over, no `system`/`assistant` roles are needed, because the model never "remembers" the first call — you're just feeding it forward yourself.

> Liu, P., et al. (2023). *Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing*. ACM Computing Surveys, 55(9), 1–35. [arXiv:2107.13586](https://arxiv.org/abs/2107.13586)

## How this works

Run the Setup cell once, then the exercise cell:

1. **Setup** loads your API keys and creates the `llm` object, same as before.
2. **The first call** (`prompt_1`) asks a narrow, well-defined sub-question — here, extracting three obligations as a plain list.
3. **The second call** (`prompt_2`) pastes `output_1` directly into a new prompt, asking the model to build the final answer around it.

Notice there's no `system` or `assistant` role anywhere — both calls are plain `user` messages, and the *only* thing connecting them is that the second prompt's text literally contains the first call's output.

In [ ]:
import os

# CrewAI's telemetry tries to reach its backend over the network on import;
# on a restricted/firewalled connection this can hang for a long time with no
# error. Disable it before crewai is imported.
os.environ.setdefault('CREWAI_DISABLE_TELEMETRY', 'true')

from dotenv import load_dotenv
from crewai import LLM
from IPython.display import Markdown, display

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

In [ ]:
TOPIC = "the EU AI Act"

# ── First call — extract, plan, or prepare ────────────────────────────────────
prompt_1 = (
    f"List the three most important obligations {TOPIC} places on companies "
    f"building AI products, one sentence each. No introduction, just the three points."
)

output_1 = llm.call(messages=[{"role": "user", "content": prompt_1}])

display(Markdown("### First call output"))
display(Markdown(output_1))

# ── Second call — use the first output to produce the final answer ────────────
# The first call's output is passed along as plain text inside this "user"
# message, not as an "assistant" turn — the two calls share no conversation history.
prompt_2 = (
    f"Using the three obligations below, write a short paragraph explaining "
    f"{TOPIC} to a 10-year-old, weaving in what companies actually have to do.\n\n"
    f"{output_1}"
)

output_2 = llm.call(messages=[{"role": "user", "content": prompt_2}])

display(Markdown("### Final output"))
display(Markdown(output_2))

## Your task

1. Run the cells (Setup first, then the exercise cell).

2. Does the two-step split (extract obligations, then explain them simply) produce a more grounded final paragraph than Step 03's single zero-shot call did? Does it still read naturally for a 10-year-old, or does it feel like two documents stapled together?

3. Try merging both prompts into one single call instead. Is the one-shot version noticeably worse, or does the model handle it fine without the split?

4. Swap `TOPIC` for your own team's topic.

5. Note what you observed — it's evidence for `EVALUATION.md`'s Section 4.3 (Prompting Techniques: Few-Shot / Chain Prompting) and Section 4.4 (Context Engineering).

## Shortcomings

Chaining calls by hand doesn't scale past two or three steps: you're manually copying text between prompts, there's no shared state beyond what you paste in, and every extra call adds latency and cost with no guarantee the split actually helped (task 3 above asks you to check exactly that). Nothing here inspects *how* the model reasoned either — only what it output at each stage.

[Step 07](step_07_chain_of_thought.ipynb) asks the model to show its reasoning explicitly, within a single call, instead of forcing you to split the task by hand.

## Resources for further reading

- Liu, P., et al. (2023). *Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing*. ACM Computing Surveys, 55(9), 1–35. [arXiv:2107.13586](https://arxiv.org/abs/2107.13586)